In [6]:
import tensorflow as tf
import numpy as np

# 禁用 eager execution 以使用 session 风格的代码
tf.compat.v1.disable_eager_execution()

# 使用 Keras 加载 MNIST 数据
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test = x_test.reshape(-1, 784).astype('float32') / 255.0
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 2000
batch_size = 100

def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1.0})
    correct_prediction = tf.equal(tf.argmax(y_pre, 1), tf.argmax(v_ys, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1.0})
    return result

def weight_variable(shape):
    initial = tf.compat.v1.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.compat.v1.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    return tf.compat.v1.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    return tf.compat.v1.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

xs = tf.compat.v1.placeholder(tf.float32, [None, 784])
ys = tf.compat.v1.placeholder(tf.float32, [None, 10])
keep_prob = tf.compat.v1.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

# 卷积层1
W_conv1 = weight_variable([7, 7, 1, 32])
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

# 卷积层2
W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

# 全连接层1
W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])
h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.compat.v1.nn.dropout(h_fc1, keep_prob)

# 全连接层2
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

# 交叉熵损失（使用 tf.math.log 替代 tf.log）
cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.math.log(prediction), axis=1))
train_step = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

def next_batch(x, y, size):
    idx = np.random.choice(len(x), size, replace=False)
    return x[idx], y[idx]

with tf.compat.v1.Session() as sess:
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)

    for i in range(max_epoch):
        batch_xs, batch_ys = next_batch(x_train, y_train, batch_size)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob: keep_prob_rate})
        if i % 100 == 0:
            acc = compute_accuracy(x_test[:1000], y_test[:1000])
            print(f"Step {i}, accuracy: {acc}")


Step 0, accuracy: 0.05299999937415123
Step 100, accuracy: 0.8569999933242798
Step 200, accuracy: 0.9070000052452087
Step 300, accuracy: 0.9359999895095825
Step 400, accuracy: 0.9459999799728394
Step 500, accuracy: 0.949999988079071
Step 600, accuracy: 0.9559999704360962
Step 700, accuracy: 0.9610000252723694
Step 800, accuracy: 0.9639999866485596
Step 900, accuracy: 0.9649999737739563
Step 1000, accuracy: 0.9739999771118164
Step 1100, accuracy: 0.9710000157356262
Step 1200, accuracy: 0.9729999899864197
Step 1300, accuracy: 0.9729999899864197
Step 1400, accuracy: 0.9729999899864197
Step 1500, accuracy: 0.9710000157356262
Step 1600, accuracy: 0.9800000190734863
Step 1700, accuracy: 0.9769999980926514
Step 1800, accuracy: 0.9789999723434448
Step 1900, accuracy: 0.9760000109672546
